# upload

Collects all `results/*.json` files and pushes them as a dataset to `notoh/qwen-coder-compilers` on HuggingFace.
Needs HF auth (`hf auth login` or `HF_TOKEN` in `.env`).

In [ ]:
import json, os
from pathlib import Path
from datasets import Dataset
from dotenv import load_dotenv

PROJECT_ROOT = Path("../.." ).resolve()
load_dotenv(PROJECT_ROOT / ".env")

RESULTS_DIR = PROJECT_ROOT / "results"
HF_REPO     = "notoh/qwen-coder-compilers"
PRIVATE     = True

In [ ]:
rows = [json.loads(p.read_text()) for p in sorted(RESULTS_DIR.glob("*.json"))]
assert rows, f"no result files found in {RESULTS_DIR}"
print(f"{len(rows)} result(s) loaded")
for r in rows:
    print(f"  {r.get('model','?')} seed={r.get('seed','?')}  exact={r.get('exact_match',float('nan')):.4f}  sim={r.get('token_sim_mean',float('nan')):.4f}")

ds = Dataset.from_list(rows)
print(ds)

info = ds.push_to_hub(HF_REPO, split="evaluation", private=PRIVATE,
                      commit_message=f"evaluation results ({len(rows)} runs)")
print("pushed:", info)